# Trabajo Práctico Aprendizaje Automático 2 - Predicción de puntuación y capitalización en texto normalizado

In [184]:
import os
import re
from datasets import load_dataset
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import unicodedata


from dataclasses import dataclass

In [185]:
# Seteamos la semilla para que los resultados sean reproducibles
SEED = 2026
np.random.seed(SEED)

In [186]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5159.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

In [187]:
def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)
    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌ El token '{token}' no pertenece al vocabulario de multilingualBERT.")
        return None
    embedding_vector = model.embeddings.word_embeddings.weight[token_id]
    
    print(f"✅ Token: '{token}' | ID: {token_id}")
    print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

## Datasets

Combinamos dos fuentes de datos en español (ambas se leen en streaming desde HuggingFace):

1. **Parlamento Europeo** (`Helsinki-NLP/europarl`): texto formal, actas parlamentarias. Las oraciones vienen en el orden original del discurso. Fuente: [Europarl](https://statmt.org/europarl/)
2. **OPUS-100** (`Helsinki-NLP/opus-100`): mezcla de varios corpus de OPUS (subtítulos, documentos técnicos, etc.), con mucho lenguaje conversacional. Las oraciones vienen mezcladas, así que dos oraciones consecutivas no tienen relación entre sí. Aporta la mayoría de las preguntas.

### Limpieza de oraciones

Cada oración pasa por tres pasos:
1. **Descarte directo** de casos donde el `.` no marca fin de oración (abreviaturas como `Sr.`, iniciales, números como `1.500`), puntos suspensivos y URLs.
2. **Mapeo de signos** que no predecimos a los que sí: `!` → `.`, `;` y `:` → `,`. Se eliminan `¡`, comillas, paréntesis, y guiones/apóstrofes que no estén dentro de una palabra.
3. **Validación**: largo razonable, solo caracteres permitidos, empieza en mayúscula, termina en `.` o `?` y tiene los `¿?` balanceados.

In [188]:
# Parámetros del dataset
N_ORACIONES_POR_FUENTE = 5000       # Oraciones limpias a leer de cada fuente
MIN_PALABRAS_ORACION = 3             # Descartamos oraciones más cortas
MAX_PALABRAS_ORACION = 60            # Descartamos oraciones más largas
MAX_ORACIONES_POR_DOCUMENTO = 20     # Oraciones consecutivas que forman un "documento"

MIN_PALABRAS_INSTANCIA = 10          # Largo mínimo de cada recorte
MAX_PALABRAS_INSTANCIA = 50          # Largo máximo de cada recorte
PROB_ORACIONES_SUELTAS = 0.15        # Proporción de documentos que se usan como oraciones sueltas

PROPORCION_TEST = 0.2              # El resto queda para train (después se hace CV sobre train)

RUTA_DATOS = "datos"                 # Carpeta donde se guarda el dataset ya armado

In [189]:
ABREVIATURAS = {
    "sr.", "sra.", "sres.", "sras.", "srta.", "dr.", "dra.", "ud.", "uds.",
    "art.", "arts.", "núm.", "nº.", "pág.", "págs.", "vol.", "cap.",
    "etc.", "ej.", "p.ej.", "aprox.", "ee.uu.", "a.c.", "d.c.",
}

CARACTERES_PERMITIDOS = set(" ¿?.,'-")


def descartar_de_entrada(texto):
    """Casos que no vale la pena intentar limpiar."""
    if "..." in texto or "…" in texto:
        return True
    if "http" in texto or "www" in texto or "@" in texto:
        return True
    # Números con punto o coma (ej: "1.500", "2,5"): el "." no sería fin de oración
    if re.search(r"\d[.,]\d", texto):
        return True
    # Abreviaturas (ej: "Sr. López"): el "." no sería fin de oración
    if any(palabra.lower() in ABREVIATURAS for palabra in texto.split()):
        return True
    # Iniciales (ej: "J. M. Barroso")
    if re.search(r"\b[A-ZÁÉÍÓÚÑ]\.\s", texto):
        return True
    return False


def mapear_signos(texto):
    """Lleva los signos que no predecimos a alguno de los que sí (¿ ? . ,) o los elimina."""
    texto = texto.replace("’", "'")
    texto = texto.replace("¡", "")
    texto = texto.replace("!", ".")
    texto = texto.replace(";", ",")
    texto = texto.replace(":", ",")
    for signo in '"«»“”()[]':
        texto = texto.replace(signo, " ")
    # Apóstrofes y guiones solo se conservan dentro de una palabra
    # (ej: "McDonald's", "franco-alemán"). Los de diálogo o incisos se eliminan.
    texto = re.sub(r"(?<!\w)['\-—–]|['\-—–](?!\w)", " ", texto)
    # Signos repetidos que pueden quedar después de mapear (ej: "?!" -> "?.")
    texto = re.sub(r"\?[.,]", "?", texto)
    texto = re.sub(r"\.+", ".", texto)
    texto = re.sub(r",+", ",", texto)
    texto = re.sub(r",\.", ".", texto)
    return texto


def normalizar_espacios_puntuacion(texto):
    # Quita espacios antes de '?', '.' o ',' (ej: "texto ?" -> "texto?")
    texto = re.sub(r"\s+([?.,])", r"\1", texto)
    # Agrega un espacio después de '?', '.' o ',' si quedaron pegados (ej: "tarde,hubo")
    texto = re.sub(r"([?.,])(?=\w)", r"\1 ", texto)
    # Quita espacios después de '¿' y agrega uno antes si quedó pegado (ej: "bien,¿no?")
    texto = re.sub(r"¿\s+", "¿", texto)
    texto = re.sub(r"(?<=\S)¿", " ¿", texto)
    # Elimina espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def es_oracion_valida(texto):
    palabras = texto.split()
    if not (MIN_PALABRAS_ORACION <= len(palabras) <= MAX_PALABRAS_ORACION):
        return False

    if any(not (c.isalnum() or c in CARACTERES_PERMITIDOS) for c in texto):
        return False

    # Cada palabra: un '¿' opcional al inicio, la palabra, y a lo sumo un signo al final
    if any(not re.fullmatch(r"¿?[\w'-]+[.,?]?", palabra) for palabra in palabras):
        return False

    # Tiene que parecer una oración completa: empieza en mayúscula y termina en . o ?
    primera_letra = next((c for c in texto if c.isalpha()), "")
    if not primera_letra.isupper():
        return False
    if not (texto.endswith(".") or texto.endswith("?")):
        return False

    # Los '¿' y '?' tienen que estar balanceados y sin anidar
    balance = 0
    for char in texto:
        if char == "¿":
            balance += 1
        elif char == "?":
            balance -= 1
        if balance < 0 or balance > 1:
            return False
    if balance != 0:
        return False

    n_letras = sum(1 for c in texto if c.isalpha())
    if n_letras < len(texto) * 0.5:
        return False

    return True


def limpiar_oracion(texto):
    """Devuelve la oración limpia, o None si hay que descartarla."""
    texto = texto.strip()
    if descartar_de_entrada(texto):
        return None
    texto = mapear_signos(texto)
    texto = normalizar_espacios_puntuacion(texto)
    if not es_oracion_valida(texto):
        return None
    return texto

In [190]:
ejemplos = [
    "Fueron los asbestos aquí. ¡Eso es lo que ocurrió!",
    "Y prefiero mudarme, ¿Entiendes?",
    "- Gracias.",
    "Sería -me parece a mí- lo mínimo, al alba del tercer milenio.",
    "Esta tarde,hubo conciertos en las ciudades ocupadas.",
    "Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.",
    "Vamos a McDonald's con el Sr. Pérez.",
    "en el apartado 2, el párrafo primero se sustituye por el texto siguiente:",
    "Perdón, he tenido una burbuja de gas extraño un momentito",
]

for ejemplo in ejemplos:
    print(f"{ejemplo!r}\n  -> {limpiar_oracion(ejemplo)!r}")

'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!'
  -> 'Fueron los asbestos aquí. Eso es lo que ocurrió.'
'Y prefiero mudarme, ¿Entiendes?'
  -> 'Y prefiero mudarme, ¿Entiendes?'
'- Gracias.'
  -> None
'Sería -me parece a mí- lo mínimo, al alba del tercer milenio.'
  -> 'Sería me parece a mí lo mínimo, al alba del tercer milenio.'
'Esta tarde,hubo conciertos en las ciudades ocupadas.'
  -> 'Esta tarde, hubo conciertos en las ciudades ocupadas.'
'Éstas ofrecen tres garantías: ningún petrolero podrá hacer escala; todas las partes deberán responder.'
  -> 'Éstas ofrecen tres garantías, ningún petrolero podrá hacer escala, todas las partes deberán responder.'
"Vamos a McDonald's con el Sr. Pérez."
  -> None
'en el apartado 2, el párrafo primero se sustituye por el texto siguiente:'
  -> None
'Perdón, he tenido una burbuja de gas extraño un momentito'
  -> None


In [191]:
def leer_documentos(nombre_dataset):
    """
    Lee el dataset en streaming y lo agrupa en documentos: listas de oraciones
    limpias que aparecen de forma consecutiva en la fuente.

    Si una oración se descarta, se corta el documento actual para no unir
    oraciones que en el original tenían texto en el medio.
    """
    stream = load_dataset(nombre_dataset, "en-es", split="train", streaming=True)

    documentos = []
    documento_actual = []
    n_oraciones = 0

    for ejemplo in stream:
        oracion = limpiar_oracion(ejemplo["translation"]["es"])

        if oracion is None:
            if documento_actual:
                documentos.append(documento_actual)
                documento_actual = []
            continue

        documento_actual.append(oracion)
        n_oraciones += 1

        if len(documento_actual) == MAX_ORACIONES_POR_DOCUMENTO:
            documentos.append(documento_actual)
            documento_actual = []

        if n_oraciones >= N_ORACIONES_POR_FUENTE:
            break

    if documento_actual:
        documentos.append(documento_actual)

    return documentos

### Armado de instancias

Ambos datasets están segmentados por oración. Si entrenáramos con oraciones completas, los modelos aprenderían un **sesgo posicional**: que la primera palabra siempre va en mayúscula y que `.` y `?` aparecen solo en la última palabra. Pero el enunciado aclara que cada instancia de evaluación puede contener *parte* de una o varias oraciones.

Para evitarlo:
1. **Documentos**: agrupamos hasta `MAX_ORACIONES_POR_DOCUMENTO` oraciones consecutivas válidas. Si una oración se descarta, el documento se corta, para no unir oraciones que en el original tenían texto en el medio. En *Europarl* esto reconstruye fragmentos reales de un mismo discurso. En *OPUS-100* las oraciones no tienen relación entre sí, pero como los modelos usan sobre todo contexto local, el cambio de tema entre oraciones no debería afectar mucho.
2. **Split por documento**: separamos train / test a nivel documento y *antes* de recortar. Así no hay fragmentos de un mismo texto en los dos splits. Cada instancia guarda su `documento_id`: al hacer CV sobre train hay que agrupar por esa columna (ej: `GroupKFold`) para que la misma regla se cumpla entre folds.
3. **Recortes**: juntamos las palabras de cada documento y las cortamos en tramos de largo aleatorio (entre `MIN_PALABRAS_INSTANCIA` y `MAX_PALABRAS_INSTANCIA`). Los tramos pueden empezar y terminar a mitad de oración. Una proporción `PROB_ORACIONES_SUELTAS` de los documentos se usa como oraciones sueltas.
4. **Fuentes separadas**: nunca se mezclan oraciones de *Europarl* y *OPUS-100* en una misma instancia.

### Etiquetas (a nivel palabra)

Por ahora trabajamos con palabras, no con tokens de BERT. Cada palabra tiene:
- `punt_inicial`: `¿` o `""`
- `punt_final`: `,`, `.`, `?` o `""`
- `capitalizacion`: 0 (minúsculas), 1 (primera letra), 2 (mixta, ej: `iPhone`), 3 (todo mayúsculas, más de una letra)

`palabra` es la entrada del modelo (minúsculas y sin signos). `palabra_original` conserva las mayúsculas, que sirve para la clase 2 (ej: armar un diccionario de palabras mixtas vistas en entrenamiento).

In [192]:
def dividir_en_splits(documentos, rng):
    """Mezcla los documentos y los reparte en train / test."""
    documentos = list(documentos)
    rng.shuffle(documentos)

    n_test = int(len(documentos) * PROPORCION_TEST)

    return {
        "test": documentos[:n_test],
        "train": documentos[n_test:],
    }


def recortar_instancias(documento, rng):
    """
    Convierte un documento en instancias (listas de palabras con puntuación y mayúsculas).

    - Con probabilidad PROB_ORACIONES_SUELTAS, cada oración es una instancia.
    - Si no, se juntan todas las oraciones y se recortan tramos de largo aleatorio,
      que pueden empezar y terminar a mitad de oración.
    """
    if rng.random() < PROB_ORACIONES_SUELTAS:
        return [oracion.split() for oracion in documento]

    palabras = " ".join(documento).split()
    instancias = []
    inicio = 0
    while inicio < len(palabras):
        largo = rng.integers(MIN_PALABRAS_INSTANCIA, MAX_PALABRAS_INSTANCIA + 1)
        instancia = palabras[inicio:inicio + largo]
        # El último tramo puede quedar muy corto: lo descartamos
        if len(instancia) >= MIN_PALABRAS_ORACION:
            instancias.append(instancia)
        inicio += largo
    return instancias


def capitalizacion(palabra):
    """0: minúsculas, 1: primera mayúscula, 2: mixta (ej: iPhone), 3: todo mayúsculas."""
    letras = [c for c in palabra if c.isalpha()]
    if not any(c.isupper() for c in letras):
        return 0
    if len(letras) > 1 and all(c.isupper() for c in letras):
        return 3
    if letras[0].isupper() and not any(c.isupper() for c in letras[1:]):
        return 1
    return 2


def etiquetar_palabra(palabra_cruda):
    """
    Recibe una palabra tal como aparece en el texto (ej: "¿Cuándo", "UBA?")
    y devuelve la palabra normalizada junto con sus etiquetas.
    """
    def char_a_int(char):
        if char == ".":
            return 1
        elif char == ",":
            return 2
        else:
            return 3
    # Puntuación inicial → 0: nada, 1: '¿'
    # Puntuación final → 0: nada, 1: '.', 2: ',', 3: '?'
    punt_inicial = 1 if palabra_cruda.startswith("¿") else 0
    ultimo_char = palabra_cruda[-1]
    punt_final = char_a_int(ultimo_char) if ultimo_char in ".,?" else 0
    palabra_original = palabra_cruda.strip("¿?.,")

    return {
        "palabra": palabra_original.lower(),
        "palabra_original": palabra_original,
        "punt_inicial": punt_inicial,
        "punt_final": punt_final,
        "capitalizacion": capitalizacion(palabra_original),
    }


def construir_dataset(documentos_por_fuente, rng):
    """
    Devuelve dos DataFrames:
    - df_instancias: una fila por instancia, con el texto original y el documento del que salió.
    - df_palabras: una fila por palabra, con las etiquetas (formato del TP, a nivel palabra).
    """
    filas_instancias = []
    filas_palabras = []
    instancia_id = 0
    documento_id = 0

    for fuente, documentos in documentos_por_fuente.items():
        splits = dividir_en_splits(documentos, rng)

        for split, documentos_split in splits.items():
            for documento in documentos_split:
                for instancia in recortar_instancias(documento, rng):
                    filas_instancias.append({
                        "instancia_id": instancia_id,
                        "documento_id": documento_id,
                        "fuente": fuente,
                        "split": split,
                        "texto": " ".join(instancia),
                    })
                    for posicion, palabra_cruda in enumerate(instancia):
                        fila = {"instancia_id": instancia_id, "posicion": posicion}
                        fila.update(etiquetar_palabra(palabra_cruda))
                        filas_palabras.append(fila)
                    instancia_id += 1
                documento_id += 1

    df_instancias = pd.DataFrame(filas_instancias)
    df_palabras = pd.DataFrame(filas_palabras)
    df_palabras = df_palabras.merge(df_instancias[["instancia_id", "documento_id", "fuente", "split"]], on="instancia_id")
    return df_instancias, df_palabras

In [193]:
ruta_instancias = os.path.join(RUTA_DATOS, "instancias.csv")
ruta_palabras = os.path.join(RUTA_DATOS, "palabras.csv")

# ACÁ SE DEFINEN df_instancias y df_palabras, ya sea cargándolos desde disco o armándolos desde cero
if os.path.exists(ruta_instancias) and os.path.exists(ruta_palabras):
    # keep_default_na=False para que la "sin puntuación" ("") no se lea como NaN
    df_instancias = pd.read_csv(ruta_instancias, keep_default_na=False)
    df_palabras = pd.read_csv(ruta_palabras, keep_default_na=False)
    print("Dataset cargado desde disco")
else:
    documentos_por_fuente = {
        "europarl": leer_documentos("Helsinki-NLP/europarl"),
        "opus100": leer_documentos("Helsinki-NLP/opus-100"),
    }
    for fuente, documentos in documentos_por_fuente.items():
        print(f"{fuente}: {len(documentos)} documentos")

    rng = np.random.default_rng(SEED)
    df_instancias, df_palabras = construir_dataset(documentos_por_fuente, rng)

    os.makedirs(RUTA_DATOS, exist_ok=True)
    df_instancias.to_csv(ruta_instancias, index=False)
    df_palabras.to_csv(ruta_palabras, index=False)
    print("Dataset armado y guardado en", RUTA_DATOS)

print(f"Instancias: {len(df_instancias)} | Palabras: {len(df_palabras)}")
print(df_instancias.groupby(["split", "fuente"]).size())

europarl: 718 documentos
opus100: 2005 documentos
Dataset armado y guardado en datos
Instancias: 7754 | Palabras: 172266
split  fuente  
test   europarl     903
       opus100      627
train  europarl    3528
       opus100     2696
dtype: int64


In [194]:
df_palabras.head(10)

,instancia_id,posicion,palabra,palabra_original,punt_inicial,punt_final,capitalizacion,documento_id,fuente,split
0,0,0,es,Es,0,0,1,0,europarl,test
1,0,1,una,una,0,0,0,0,europarl,test
2,0,2,estructura,estructura,0,0,0,0,europarl,test
3,0,3,respecto,respecto,0,0,0,0,europarl,test
4,0,4,de,de,0,0,0,0,europarl,test
5,0,5,la,la,0,0,0,0,europarl,test
6,0,6,cual,cual,0,0,0,0,europarl,test
7,0,7,la,la,0,0,0,0,europarl,test
8,0,8,unión,Unión,0,0,1,0,europarl,test
9,0,9,europea,Europea,0,0,1,0,europarl,test


### Chequeo de clases

In [195]:
# Distribución de clases a nivel palabra (que es como se evalúa), por fuente
for columna in ["punt_inicial", "punt_final", "capitalizacion"]:
    tabla = pd.crosstab(df_palabras[columna], df_palabras["fuente"], margins=True)
    porcentajes = pd.crosstab(df_palabras[columna], df_palabras["fuente"], normalize="columns").round(4) * 100
    print(f"--- {columna} (cantidad de palabras) ---")
    print(tabla)
    print(f"--- {columna} (% por fuente) ---")
    print(porcentajes, "\n")

--- punt_inicial (cantidad de palabras) ---
fuente        europarl  opus100     All
punt_inicial                           
0               118891    52333  171224
1                  218      824    1042
All             119109    53157  172266
--- punt_inicial (% por fuente) ---
fuente        europarl  opus100
punt_inicial                   
0                99.82    98.45
1                 0.18     1.55 

--- punt_final (cantidad de palabras) ---
fuente      europarl  opus100     All
punt_final                           
0             107115    44996  152111
1               4818     4430    9248
2               6959     2918    9877
3                217      813    1030
All           119109    53157  172266
--- punt_final (% por fuente) ---
fuente      europarl  opus100
punt_final                   
0              89.93    84.65
1               4.05     8.33
2               5.84     5.49
3               0.18     1.53 

--- capitalizacion (cantidad de palabras) ---
fuente          euro

In [196]:
# Chequeo del sesgo posicional: ¿los signos de cierre siguen apareciendo solo al final?
ultima_posicion = df_palabras.groupby("instancia_id")["posicion"].transform("max")
es_ultima = df_palabras["posicion"] == ultima_posicion
es_primera = df_palabras["posicion"] == 0

for signo in [".", "?"]:
    con_signo = df_palabras["punt_final"] == signo
    intermedios = (con_signo & ~es_ultima).sum()
    print(f"'{signo}': {con_signo.sum()} en total, {intermedios} en el medio de una instancia "
          f"({intermedios / con_signo.sum():.1%})")

termina_cortada = es_ultima & ~df_palabras["punt_final"].isin([".", "?"])
empieza_minuscula = es_primera & (df_palabras["capitalizacion"] == 0)
print(f"Instancias que terminan a mitad de oración: {termina_cortada.sum() / len(df_instancias):.1%}")
print(f"Instancias que empiezan en minúscula: {empieza_minuscula.sum() / len(df_instancias):.1%}")

'.': 0 en total, 0 en el medio de una instancia (nan%)
'?': 0 en total, 0 en el medio de una instancia (nan%)
Instancias que terminan a mitad de oración: 100.0%
Instancias que empiezan en minúscula: 46.4%


/tmp/ipykernel_16454/3473395508.py:10: RuntimeWarning: invalid value encountered in scalar divide
  f"({intermedios / con_signo.sum():.1%})")


### Ejemplos del Dataset

In [221]:
def map_numerito_a_punt_inical(numerito):
    if numerito == 0:
        return ""
    elif numerito == 1:
        return "¿"
def map_numerito_a_punt_final(numerito):
    if numerito == 0:
        return ""
    elif numerito == 1:
        return "."
    elif numerito == 2:
        return ","
    return "?"


def reconstruir_texto(palabras_instancia):
    """Arma el texto con puntuación y mayúsculas a partir de las etiquetas de cada palabra."""
    partes = []
    for _, fila in palabras_instancia.iterrows():
        palabra = fila["palabra"]
        if fila["capitalizacion"] == 1:
            palabra = palabra[0].upper() + palabra[1:]
        elif fila["capitalizacion"] == 2:
            palabra = fila["palabra_original"]  # no hay forma de deducir qué letras van en mayúscula
        elif fila["capitalizacion"] == 3:
            palabra = palabra.upper()
        partes.append(map_numerito_a_punt_inical(fila["punt_inicial"]) + palabra + map_numerito_a_punt_final(fila["punt_final"]))
    return " ".join(partes)


# Chequeo: reconstruir el texto desde las etiquetas tiene que devolver el original
muestra = df_instancias.sample(500, random_state=SEED)
palabras_muestra = df_palabras[df_palabras["instancia_id"].isin(muestra["instancia_id"])]
errores = 0
for _, instancia in muestra.iterrows():
    palabras_instancia = palabras_muestra[palabras_muestra["instancia_id"] == instancia["instancia_id"]]
    if reconstruir_texto(palabras_instancia) != instancia["texto"]:
        errores += 1
print(f"Reconstrucciones distintas al original: {errores} de {len(muestra)}\n")

for fuente in ["europarl", "opus100"]:
    print(f"Ejemplos de {fuente}:")
    for texto in df_instancias[df_instancias["fuente"] == fuente]["texto"].head(4):
        pass
        #print(f"  → {texto}")
    print()

Reconstrucciones distintas al original: 0 de 500

Ejemplos de europarl:

Ejemplos de opus100:



In [222]:
oraicon = "fui al taller y no queria comi una hamburguesa vos"
dfff = pd.DataFrame
reconstruir_texto(oraicon)

AttributeError: 'str' object has no attribute 'iterrows'

# Herramientas comunes a todos los modelos

Todo lo de esta sección sirve para cualquier modelo (clásicos con o sin embeddings, RNN, etc.), siempre que se escriban dos funciones:

- `entrenar(df_train) -> modelos`: recibe palabras etiquetadas (como `df_palabras`) y devuelve lo que haga falta para predecir.
- `predecir(modelos, df) -> DataFrame`: recibe palabras normalizadas (columnas `instancia_id`, `posicion`, `palabra`) y devuelve un DataFrame **con el mismo índice que `df`** y una columna por tarea (`punt_inicial`, `punt_final`, `capitalizacion`).

Con eso se pueden usar `validacion_cruzada`, `evaluar`, `etiquetar_parrafo` y `comparar_parrafo`.

Para los modelos clásicos (una predicción por palabra a partir de un vector de atributos) además está armado el pipeline completo: solo hay que pasarle la función que extrae los atributos y la que crea el clasificador.

In [198]:
TAREAS = ["punt_final", "capitalizacion", "punt_inicial"]

CLASES = {
    "punt_inicial": ["¿", ""],
    "punt_final": [",", ".", "?", ""],
    "capitalizacion": [0, 1, 2, 3],
}

# Parámetros de la validación
N_FOLDS = 1                  # 1: una sola partición train / validación. >1: k-fold
PROPORCION_VALIDACION = 0.2  # Solo se usa si N_FOLDS == 1
COLUMNA_GRUPO = "documento_id"  # Lo que nunca se separa entre train y validación: "documento_id" o "instancia_id"

# Columnas de df_palabras que no son atributos (identificadores y etiquetas)
COLUMNAS_NO_ATRIBUTOS = [
    "instancia_id", "documento_id", "fuente", "split", "palabra", "palabra_original",
    "punt_inicial", "punt_final", "capitalizacion",
]

df_train = df_palabras[df_palabras["split"] == "train"]
df_test = df_palabras[df_palabras["split"] == "test"]


def columnas_de_atributos(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in COLUMNAS_NO_ATRIBUTOS]

In [199]:
def particiones_train_validacion(df, n_folds):
    """
    Particiones train / validación con StratifiedGroupKFold:
    - las palabras de un mismo grupo (COLUMNA_GRUPO) quedan siempre del mismo lado,
    - y se intenta mantener la proporción de clases de punt_final en cada fold.

    n_folds == 1: una sola partición, tomando el primer fold de un k-fold con
    k = 1 / PROPORCION_VALIDACION (ej: 0.2 -> 5 folds -> 20% de validación).

    Devuelve una lista de pares (índices de entrenamiento, índices de validación).
    """
    k = round(1 / PROPORCION_VALIDACION) if n_folds == 1 else n_folds
    cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)

    particiones = []
    for idx_entrenamiento, idx_validacion in cv.split(df, df["punt_final"], groups=df[COLUMNA_GRUPO]):
        particiones.append((df.index[idx_entrenamiento], df.index[idx_validacion]))
        if n_folds == 1:
            break
    return particiones

Pipeline para los modelos clasicos donde hay un modelo para cada clase que queremos predecir

In [200]:
# Pipeline para modelos clásicos: un clasificador por tarea, todos con los mismos atributos.
# extraer_atributos(df) tiene que devolver df con columnas de atributos agregadas
# (conservando el índice), y nuevo_modelo() un clasificador de sklearn sin entrenar.
def entrenar(X, ys, nuevo_modelo):
    modelos = {}
    for target in ys.columns:
        modelo = nuevo_modelo()
        modelo.fit(X, ys[target])
        modelos[target] = modelo
    return modelos

def validar(X_test, ys_test, modelos):
    resultados = {}
    print(ys_test)
    for target in ys_test.columns:
        y_pred = modelos[target].predict(X_test)
        f1 = f1_score(ys_test[target], y_pred, average=None)
        resultados[target] = f1
    return resultados

def predecir_pipeline_clasico(modelos, df):
    """
    Recibe palabras normalizadas (columnas instancia_id, posicion, palabra) y devuelve un
    DataFrame con las predicciones de las tres tareas, con el mismo índice que df.
    """
    X = modelos["extraer_atributos"](df)

    resultado = X[["instancia_id", "posicion", "palabra"]].copy()
    for tarea in TAREAS:
        resultado[tarea] = modelos[tarea].predict(X[modelos["columnas"]])
    return resultado

Todo esto se puede usar si se sigue la misma interfaz propuesta:

In [201]:
@dataclass
class InformeCompleto:
    informe: pd.DataFrame
    y_global_true: np.ndarray
    y_global_pred: np.ndarray

def classfication_report_dict(modelo, y_pred, y_true): 
    
    """
    Devuelve un diccionario con las métricas de clasificación para cada clase y el promedio macro.
    """
    
    y_pred = modelo.predict(y_true)
    report = f1_score(y_true, y_pred, labels=TAREAS, average=None, zero_division=0)
    report_dict = {f"F1 '{label}'": f1 for label, f1 in zip(TAREAS, report)}
    report_dict["F1 macro"] = f1_score(y_true, y_pred, labels=TAREAS, average="macro", zero_division=0)
    return report_dict

def metricas(y_true, y_pred, tarea, conjunto="val."):
    """F1 macro (como se evalúa en el TP) y F1 de cada clase."""
    clases = CLASES[tarea]
    f1_por_clase = f1_score(y_true, y_pred, labels=clases, average=None, zero_division=0)
    fila = {f"F1 macro ({conjunto})": f1_score(y_true, y_pred, labels=clases, average="macro", zero_division=0)}
    for clase, f1 in zip(clases, f1_por_clase):
        fila[f"F1 '{clase}' ({conjunto})"] = f1
    return fila


def validacion_cruzada(df, n_folds, entrenar, predecir):
    """
    Entrena y evalúa un modelo en cada partición de df (ver particiones_train_validacion).
    entrenar y predecir son las funciones del modelo (ver la descripción de la sección).
    Devuelve un InformeCompleto por tarea.
    """
    filas = {tarea: [] for tarea in TAREAS}
    predicciones_validacion = []

    for fold, (idx_entrenamiento, idx_validacion) in enumerate(particiones_train_validacion(df, n_folds)):
        df_entrenamiento = df.loc[idx_entrenamiento]
        df_validacion = df.loc[idx_validacion]

        modelos = entrenar(df_entrenamiento)
        pred_entrenamiento = predecir(modelos, df_entrenamiento)
        pred_validacion = predecir(modelos, df_validacion)
        predicciones_validacion.append(pred_validacion)

        for tarea in TAREAS:
            fila = {
                "Permutación": str(fold + 1),
                "F1 macro (train)": f1_score(
                    df_entrenamiento.loc[pred_entrenamiento.index, tarea], pred_entrenamiento[tarea],
                    labels=CLASES[tarea], average="macro", zero_division=0,
                ),
            }
            fila.update(metricas(df_validacion.loc[pred_validacion.index, tarea], pred_validacion[tarea], tarea))
            filas[tarea].append(fila)

        print(f"Fold {fold + 1} listo")

    pred_global = pd.concat(predicciones_validacion)

    informes = {}
    for tarea in TAREAS:
        y_global_true = df.loc[pred_global.index, tarea].to_numpy()
        y_global_pred = pred_global[tarea].to_numpy()

        df_tabla = pd.DataFrame(filas[tarea])
        if n_folds > 1:
            promedios = df_tabla.drop(columns=["Permutación"]).mean()
            df_tabla.loc[len(df_tabla)] = ["Promedios"] + list(promedios)
            df_tabla.loc[len(df_tabla)] = ["Global", "(NO)"] + list(metricas(y_global_true, y_global_pred, tarea).values())

        informes[tarea] = InformeCompleto(
            informe=df_tabla,
            y_global_true=y_global_true,
            y_global_pred=y_global_pred,
        )

    return informes


def mostrar_informes(informes):
    for tarea in TAREAS:
        print(f"\n=== {tarea} ===")
        display(informes[tarea].informe)


def mostrar_matrices_confusion(informes):
    """Filas: clase real, columnas: clase predicha. Sobre todas las palabras de validación."""
    for tarea in TAREAS:
        print(f"\n=== {tarea} ===")
        display(pd.crosstab(
            pd.Series(informes[tarea].y_global_true, name="real"),
            pd.Series(informes[tarea].y_global_pred, name="predicha"),
        ))


def evaluar(modelos, predecir, df, conjunto="test"):
    """F1 de cada tarea sobre un conjunto ya etiquetado (ej: df_test). Correr test solo al final."""
    pred = predecir(modelos, df)
    for tarea in TAREAS:
        resultado = metricas(df.loc[pred.index, tarea], pred[tarea], tarea, conjunto=conjunto)
        print(f"\n=== {tarea} ===")
        display(pd.DataFrame([resultado]))

In [202]:
def normalizar_parrafo(texto):
    """Pasa a minúsculas, saca los signos de puntuación y separa en palabras."""
    texto = re.sub(r"[¿?¡!.,;:\"«»()]", " ", texto.lower())
    return texto.split()


def texto_predicho(prediccion):
    """Reconstruye el texto a partir de las predicciones."""
    prediccion = prediccion.copy()
    # Para la clase 2 (ej: "McDonald's") no sabemos qué letras van en mayúscula: ponemos solo la primera
    prediccion["palabra_original"] = prediccion["palabra"].str.capitalize()
    return reconstruir_texto(prediccion)


def etiquetar_parrafo(modelos, predecir, texto):
    """Normaliza un párrafo, lo pasa por los modelos y reconstruye el texto."""
    palabras = normalizar_parrafo(texto)
    df = pd.DataFrame({
        "instancia_id": 0,
        "posicion": range(len(palabras)),
        "palabra": palabras,
    })
    return texto_predicho(predecir(modelos, df))


def comparar_parrafo(modelos, predecir, texto_real):
    """
    Recibe un párrafo con puntuación y mayúsculas, lo normaliza, lo predice y compara
    el texto real con el predicho, palabra por palabra.
    """
    palabras = normalizar_espacios_puntuacion(texto_real).split()
    real = pd.DataFrame([etiquetar_palabra(palabra) for palabra in palabras])
    real["instancia_id"] = 0
    real["posicion"] = range(len(real))

    pred = predecir(modelos, real[["instancia_id", "posicion", "palabra"]])

    print(f"Real:     {reconstruir_texto(real)}")
    print(f"Predicho: {texto_predicho(pred)}")

    comparacion = pd.DataFrame({"palabra": real["palabra"]})
    for tarea in TAREAS:
        comparacion[f"{tarea} (real)"] = real[tarea]
        comparacion[f"{tarea} (pred.)"] = pred.loc[real.index, tarea]
    comparacion["errores"] = [
        ", ".join(tarea for tarea in TAREAS if real.loc[i, tarea] != pred.loc[i, tarea])
        for i in real.index
    ]
    display(comparacion)

# Random Forest (atributos a mano, sin embeddings)

Entrenamos un Random Forest independiente por tarea, todos con los mismos atributos de las palabras (y de sus vecinas):

```
                          ┌─► RF punt_final (, . ? "")
palabras ─► atributos ────┼─► RF capitalizacion (0 1 2 3)
                          └─► RF punt_inicial (¿ "")
```

Usa el pipeline clásico de la sección anterior: `añadir_atributos` extrae los atributos y `nuevo_random_forest` crea cada clasificador.

## Atributos

In [203]:
VOCALES = set("aeiouáéíóúü")
VOCALES_SIN_TILDE = set("aeiouü")

ADVERBIOS_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cómo", "cuándo", "dónde",
    "cuánto", "cuánta", "cuántos", "cuántas", "cuál", "cuáles",
}

PRONOMBRES_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cuál", "cuáles", "cuánto",
    "cuánta", "cuántos", "cuántas",
}

# Unimos ambos para el atributo "es_palabra_interrogativa"
PALABRAS_INTERROGATIVAS = ADVERBIOS_INTERROGATIVOS | PRONOMBRES_INTERROGATIVOS

# Palabras que casi nunca cierran una oración (artículos, preposiciones, posesivos)
PALABRAS_FUNCIONALES = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "lo",
    "de", "del", "a", "al", "en", "con", "por", "para", "sin", "sobre",
    "entre", "hacia", "desde", "hasta", "según",
    "mi", "tu", "su", "mis", "tus", "sus",
    "nuestro", "nuestra", "nuestros", "nuestras",
}

# Palabras que suelen abrir una cláusula nueva (antes suele ir una coma o un punto)
CONECTORES = {
    "pero", "sino", "aunque", "porque", "pues", "entonces", "además",
    "embargo", "mientras", "cuando", "donde", "como", "que", "y", "o", "ni",
}

def _es_palabra_interrogativa(palabra: str) -> int:
    return int(palabra in PALABRAS_INTERROGATIVAS)

In [204]:
# Atributos de las palabras vecinas que se agregan como contexto
COLUMNAS_CONTEXTO = [
    "es_palabra_interrogativa", "tiene_tilde", "es_funcional", "es_conector", "cantidad_letras",
]


def _tiene_tilde(palabra: str) -> int:
    """1 si la palabra contiene al menos una vocal con tilde (á é í ó ú)."""
    return int(any(ch in "áéíóú" for ch in palabra))


def _cantidad_vocales(palabra: str) -> int:
    return sum(1 for ch in palabra if ch in VOCALES)


def _cantidad_consonantes(palabra: str) -> int:
    """Cuenta letras que no son vocales. Ignora dígitos, espacios, etc."""
    return sum(
        1 for ch in palabra
        if unicodedata.category(ch).startswith("L") and ch not in VOCALES
    )


def extraer_atributos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega atributos léxicos y contextuales a un DataFrame a nivel palabra
    (como df_palabras). Conserva el índice de df.

    Solo se usan atributos que se pueden calcular sobre el texto normalizado:
    nada que dependa de las etiquetas, porque en inferencia no las tenemos.
    """
    df = df.copy()

    columnas_requeridas = ["instancia_id", "posicion", "palabra"]
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en el DataFrame: {faltantes}")

    # Orden explícito para que los shifts prev/next sean correctos.
    df = df.sort_values(["instancia_id", "posicion"])

    # Atributos por palabra
    df["tiene_tilde"] = df["palabra"].map(_tiene_tilde)
    df["cantidad_vocales"] = df["palabra"].map(_cantidad_vocales)
    df["cantidad_consonantes"] = df["palabra"].map(_cantidad_consonantes)
    df["cantidad_letras"] = df["cantidad_vocales"] + df["cantidad_consonantes"]
    df["es_palabra_interrogativa"] = df["palabra"].map(_es_palabra_interrogativa)
    df["es_funcional"] = df["palabra"].isin(PALABRAS_FUNCIONALES).astype(int)
    df["es_conector"] = df["palabra"].isin(CONECTORES).astype(int)
    df["tiene_digitos"] = df["palabra"].str.contains(r"\d").astype(int)
    df["termina_en_mente"] = df["palabra"].str.endswith("mente").astype(int)
    df["termina_en_r"] = df["palabra"].str.endswith("r").astype(int)  # infinitivos

    # Posición dentro de la instancia
    df["posicion_desde_final"] = df.groupby("instancia_id")["posicion"].transform("max") - df["posicion"]

    # Contexto: atributos de las 2 palabras anteriores y las 2 siguientes dentro de la
    # misma instancia. -1 indica que no hay palabra (inicio o fin de la instancia).
    for col in COLUMNAS_CONTEXTO:
        for d in [1, 2]:
            df[f"prev{d}_{col}"] = df.groupby("instancia_id")[col].shift(d).fillna(-1)
            df[f"next{d}_{col}"] = df.groupby("instancia_id")[col].shift(-d).fillna(-1)

    return df

In [205]:
df_train

,instancia_id,posicion,palabra,palabra_original,punt_inicial,punt_final,capitalizacion,documento_id,fuente,split
25206,903,0,señora,Señora,0,0,1,143,europarl,train
25207,903,1,presidenta,Presidenta,0,2,1,143,europarl,train
25208,903,2,señorías,Señorías,0,2,1,143,europarl,train
25209,903,3,hoy,hoy,0,0,0,143,europarl,train
25210,903,4,tenemos,tenemos,0,0,0,143,europarl,train
...,...,...,...,...,...,...,...,...,...,...
172261,7753,2,hablamos,hablamos,0,1,0,2722,opus100,train
172262,7753,3,un,Un,1,0,1,2722,opus100,train
172263,7753,4,predecesor,predecesor,0,0,0,2722,opus100,train
172264,7753,5,del,del,0,0,0,2722,opus100,train


In [206]:
# Prueba: atributos de las primeras 10 instancias de train
ids_10 = df_train["instancia_id"].unique()[:10]
df_etiquetado = extraer_atributos(df_train[df_train["instancia_id"].isin(ids_10)])

print(f"Filas etiquetadas: {len(df_etiquetado)} | Atributos: {len(columnas_de_atributos(df_etiquetado))}")
df_etiquetado.head(20)

Filas etiquetadas: 276 | Atributos: 32


,instancia_id,posicion,palabra,palabra_original,punt_inicial,punt_final,capitalizacion,documento_id,fuente,split,...,prev2_es_funcional,next2_es_funcional,prev1_es_conector,next1_es_conector,prev2_es_conector,next2_es_conector,prev1_cantidad_letras,next1_cantidad_letras,prev2_cantidad_letras,next2_cantidad_letras
25206,903,0,señora,Señora,0,0,1,143,europarl,train,...,-1.0,0.0,-1.0,0.0,-1.0,0.0,-1.0,10.0,-1.0,8.0
25207,903,1,presidenta,Presidenta,0,2,1,143,europarl,train,...,-1.0,0.0,0.0,0.0,-1.0,0.0,6.0,8.0,-1.0,3.0
25208,903,2,señorías,Señorías,0,2,1,143,europarl,train,...,0.0,0.0,0.0,0.0,0.0,0.0,10.0,3.0,6.0,7.0
25209,903,3,hoy,hoy,0,0,0,143,europarl,train,...,0.0,0.0,0.0,0.0,0.0,0.0,8.0,7.0,10.0,4.0
25210,903,4,tenemos,tenemos,0,0,0,143,europarl,train,...,0.0,1.0,0.0,0.0,0.0,0.0,3.0,4.0,8.0,3.0
25211,903,5,cita,cita,0,0,0,143,europarl,train,...,0.0,1.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,2.0
25212,903,6,con,con,0,0,0,143,europarl,train,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,7.0,10.0
25213,903,7,la,la,0,0,0,143,europarl,train,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,10.0,4.0,7.0
25214,903,8,hipocresía,hipocresía,0,1,0,143,europarl,train,...,1.0,1.0,0.0,0.0,0.0,0.0,2.0,7.0,3.0,2.0
25215,903,9,tenemos,Tenemos,0,0,1,143,europarl,train,...,1.0,0.0,0.0,0.0,0.0,0.0,10.0,2.0,2.0,7.0


## Modelo

In [207]:
df_train_clasico = (extraer_atributos(df_train))


In [208]:
cols_attrs = columnas_de_atributos(df_train_clasico)
X = df_train_clasico[columnas_de_atributos]
ys = df_train_clasico[TAREAS]

In [209]:
ys.columns

Index(['punt_final', 'capitalizacion', 'punt_inicial'], dtype='str')

In [210]:
#rfs = entrenar_pipeline(X, y, extraer_atributos, nuevo_random_forest)

In [211]:
modelos_rf = entrenar(X, ys, nuevo_random_forest)

In [214]:
df_test_clasico = extraer_atributos(df_test)

In [215]:
X_test = df_test_clasico[cols_attrs]
ys_test = df_test_clasico[TAREAS]

In [216]:
validar(X_test, ys_test, modelos_rf)

        punt_final  capitalizacion  punt_inicial
0                0               1             0
1                0               0             0
2                0               0             0
3                0               0             0
4                0               0             0
...            ...             ...           ...
128872           3               0             0
128873           0               1             0
128874           0               0             0
128875           0               0             0
128876           1               0             0

[34974 rows x 3 columns]


{'punt_final': array([0.86766973, 0.3398533 , 0.25346832, 0.19607843]),
 'capitalizacion': array([0.90785808, 0.46824284, 0.        , 0.26344086]),
 'punt_inicial': array([0.99226487, 0.31673052])}

In [217]:
# Hiperparámetros de los Random Forest
PARAMS_RF = {
    "n_estimators": 100,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",  # las clases ¿ ? y la capitalización 2 y 3 son muy raras
}


def nuevo_random_forest():
    return RandomForestClassifier(**PARAMS_RF, random_state=SEED, n_jobs=-1)


def entrenar_rf(df_train):
    return entrenar_pipeline_clasico(df_train, extraer_atributos, nuevo_random_forest)

In [176]:
df_train_svm = df_train.copy()


In [177]:
def extraer_atributos_embeddings(
    df_train,
    tokenizer,
    max_tokens: int = 512,
    ventana: int | None = None,
):
    """
    Construye una lista de instancias token-a-token para predecir '¿'
    a partir de texto normalizado.

    Cada instancia es un dict:
        {
            'oracion_id': int,
            'token': str,
            'token_id': int,
            'distancia_inicio': int,   # cuántos tokens no-'¿' hay antes
            'label': int,              # 1 si estaba precedido por '¿'
            'ventana': list[int|None], # solo si ventana no es None
        }
    """
    df_train['prom_emb'] = df_train['palabras'].apply(lambda x: np.mean(model.embeddings.word_embeddings(tokenizer.tokenize(x))))

    ultimo = -999
    for instancia in df_train.iterrows():
        da
        # 1) Tokenizar la oración original (con su puntuación)
        tokens = tokenizer.tokenize(oracion)
        if not tokens:
            continue

        # 2) Truncar al límite de BERT
        tokens = tokens[:max_tokens]

        # 3) Construir input limpio + etiquetas
        tokens = []
        target = []
        for i, tok in enumerate(tokens):
            tokens.append(tok)
            # etiqueta: ¿el token anterior (en el original) era '¿'?
            target.append(1 if (i > 0 and tokens[i - 1] == "¿") else 0)

        # Por si no habían tokens ¿?
        if not tokens:
            continue

        # 4) Convertir a ids (BERT ya tiene embeddings: solo pasamos input_ids)
        ids = tokenizer.convert_tokens_to_ids(tokens)

        # 5) Armar registros (con o sin ventana de contexto)
        n = len(ids)
        for i in range(n):
            registro = {
                "oracion_id": oracion_id, # esto no lo había pensado pero se ve bien...
                "token": tokens[i],
                "token_id": ids[i],
                "distancia_inicio": i,
                "label": target[i],
            }
            if ventana is not None:
                vecinos = [
                    ids[j] if 0 <= j < n else None
                    for j in range(i - ventana, i + ventana + 1)
                ]
                registro["ventana"] = vecinos

In [178]:
dim = model.config.hidden_size
df_train_svm = df_train_svm.sort_values(["instancia_id",
"posicion"]).reset_index(drop=True)

emb_palabra = np.zeros((len(df_train_svm), dim), dtype=np.float32)
emb_ventana = np.zeros((len(df_train_svm), dim), dtype=np.float32)

palabras_col = df_palabras["palabra"].values
for inst_id, idx in df_palabras.groupby("instancia_id").indices.items():
    emb = np.stack([get_multilingual_token_embedding(p) for p in palabras_col[idx]]) # (n, 768)
    n = len(emb)

    # promedio de [anterior, actual, siguiente]; en los bordes solo con los vecinos que existen
    suma = emb.copy()
    cnt = np.ones((n, 1), dtype=np.float32)
    suma[1:]  += emb[:-1]; cnt[1:]  += 1   # palabra anterior
    suma[:-1] += emb[1:];  cnt[:-1] += 1   # palabra siguiente

    emb_palabra[idx] = emb
    emb_ventana[idx] = suma / cnt

df_train_svm["emb_palabra"] = list(emb_palabra)    # embedding de la palabra sola
df_train_svm["emb_ventana"] = list(emb_ventana)    # promedio anterior+ actual + siguient

✅ Token: 'es' | ID: 10196
Embedding shape: torch.Size([768])
✅ Token: 'una' | ID: 10153
Embedding shape: torch.Size([768])
✅ Token: 'estructura' | ID: 21795
Embedding shape: torch.Size([768])
✅ Token: 'respecto' | ID: 30989
Embedding shape: torch.Size([768])
✅ Token: 'de' | ID: 10104
Embedding shape: torch.Size([768])
✅ Token: 'la' | ID: 10109
Embedding shape: torch.Size([768])
✅ Token: 'cual' | ID: 12891
Embedding shape: torch.Size([768])
✅ Token: 'la' | ID: 10109
Embedding shape: torch.Size([768])
✅ Token: 'unión' | ID: 44064
Embedding shape: torch.Size([768])
✅ Token: 'europea' | ID: 30908
Embedding shape: torch.Size([768])
✅ Token: 'ha' | ID: 10228
Embedding shape: torch.Size([768])
✅ Token: 'dado' | ID: 25665
Embedding shape: torch.Size([768])
✅ Token: 'el' | ID: 10125
Embedding shape: torch.Size([768])
✅ Token: 'apoyo' | ID: 30390
Embedding shape: torch.Size([768])
✅ Token: 'posible' | ID: 22777
Embedding shape: torch.Size([768])
✅ Token: 'y' | ID: 193
Embedding shape: torch.Size

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [38]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.model_selection import (cross_validate, StratifiedKFold, ParameterGrid, RandomizedSearchCV)
stratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def mejor_SVC_dado_df():
    param_dist = [
        {
            'estimator__kernel': ['rbf'],
            'estimator__C': np.logspace(-3, 3, 7), 
            'estimator__gamma': ['scale', 'auto'], 
        }
    ]

    modelo_base = CalibratedClassifierCV(SVC(), ensemble=False)

    random_search_svc = RandomizedSearchCV(
        modelo_base,
        param_distributions=param_dist,
        n_iter=1,
        scoring='roc_auc',
        cv=stratifiedKFold,
        random_state=SEED,
        n_jobs=-1
    )
    
    return random_search_svc   

In [ ]:
#modelo_svc = entrenar_pipeline_clasico(df_train, extraer_atributos, mejor_SVC_dado_df)

## Validación

In [ ]:
informes_rf = validacion_cruzada(df_train, N_FOLDS, entrenar_rf, predecir_pipeline_clasico)
mostrar_informes(informes_rf)
# por qué solo muestra el fold 1 ¿?
# la validación no debería entrenar un modelo, cambiarlo

Fold 1 listo

=== punt_final ===


,Permutación,F1 macro (train),F1 macro (val.),"F1 ',' (val.)",F1 '.' (val.),F1 '?' (val.),F1 '' (val.)
0,1,0.621475,0.417867,0.254701,0.346778,0.207031,0.862956



=== capitalizacion ===


,Permutación,F1 macro (train),F1 macro (val.),F1 '0' (val.),F1 '1' (val.),F1 '2' (val.),F1 '3' (val.)
0,1,0.756606,0.401926,0.909285,0.466536,0.0,0.231884



=== punt_inicial ===


,Permutación,F1 macro (train),F1 macro (val.),F1 '¿' (val.),F1 '' (val.)
0,1,0.730772,0.650193,0.308642,0.991743


In [27]:
mostrar_matrices_confusion(informes_rf)


=== punt_final ===


predicha,,",",.,?
real,,,,
,19001,3702,1409,111
",",528,833,205,12
.,251,404,662,173
?,34,24,52,53



=== capitalizacion ===


predicha,0,1,2,3
real,,,,
0,21325,2818,1,196
1,1203,1781,0,23
2,15,6,0,1
3,22,23,0,40



=== punt_inicial ===


predicha,,¿
real,,
,26906,384
¿,64,100


In [ ]:

# Abrir el archivo en modo binario de lectura ('rb')
import joblib


with open('modelo_final_0_SVC.pkl', 'r+b' if False else 'rb') as archivo:
    modelo = joblib.load(archivo)

In [ ]:
df_test.head(50)

,instancia_id,posicion,palabra,palabra_original,punt_inicial,punt_final,capitalizacion,documento_id,fuente,split
0,0,0,es,Es,,,1,0,europarl,test
1,0,1,una,una,,,0,0,europarl,test
2,0,2,estructura,estructura,,,0,0,europarl,test
3,0,3,respecto,respecto,,,0,0,europarl,test
4,0,4,de,de,,,0,0,europarl,test
5,0,5,la,la,,,0,0,europarl,test
6,0,6,cual,cual,,,0,0,europarl,test
7,0,7,la,la,,,0,0,europarl,test
8,0,8,unión,Unión,,,1,0,europarl,test
9,0,9,europea,Europea,,,1,0,europarl,test


In [27]:
df_train

,instancia_id,posicion,palabra,palabra_original,punt_inicial,punt_final,capitalizacion,documento_id,fuente,split
25206,903,0,señora,Señora,,,1,143,europarl,train
25207,903,1,presidenta,Presidenta,,",",1,143,europarl,train
25208,903,2,señorías,Señorías,,",",1,143,europarl,train
25209,903,3,hoy,hoy,,,0,143,europarl,train
25210,903,4,tenemos,tenemos,,,0,143,europarl,train
...,...,...,...,...,...,...,...,...,...,...
172261,7753,2,hablamos,hablamos,,.,0,2722,opus100,train
172262,7753,3,un,Un,¿,,1,2722,opus100,train
172263,7753,4,predecesor,predecesor,,,0,2722,opus100,train
172264,7753,5,del,del,,,0,2722,opus100,train


In [ ]:
# evaluar SVC
from sklearn.metrics import classification_report
X_test = extraer_atributos(df_test)
y_test = df_test[df_test["split"] == "test"][TAREAS]

y_pred = modelo.predict(X_test)
classfication_report_dict(modelo, y_pred, y_test)

NameError: name 'VOCALES' is not defined

## Pipeline completo

Entrenamos con todo train y lo usamos para etiquetar párrafos nuevos.

In [28]:
# Entrenamos los modelos con todo train
modelos_rf = entrenar_rf(df_train)

# Atributos más importantes de cada modelo
for tarea in TAREAS:
    importancias = pd.Series(modelos_rf[tarea].feature_importances_, index=modelos_rf["columnas"])
    print(f"\n=== {tarea} ===")
    print(importancias.sort_values(ascending=False).head(10).round(3).to_string())


=== punt_final ===
posicion                 0.091
es_funcional             0.091
posicion_desde_final     0.082
next1_cantidad_letras    0.080
cantidad_letras          0.071
cantidad_vocales         0.062
prev2_cantidad_letras    0.058
next2_cantidad_letras    0.057
prev1_cantidad_letras    0.057
cantidad_consonantes     0.053

=== capitalizacion ===
cantidad_letras          0.124
cantidad_consonantes     0.113
cantidad_vocales         0.079
prev1_cantidad_letras    0.076
posicion_desde_final     0.074
posicion                 0.073
next1_cantidad_letras    0.062
prev2_cantidad_letras    0.054
next2_cantidad_letras    0.054
es_funcional             0.051

=== punt_inicial ===
prev1_cantidad_letras       0.097
es_palabra_interrogativa    0.093
posicion                    0.076
prev1_es_funcional          0.067
posicion_desde_final        0.062
cantidad_letras             0.050
tiene_tilde                 0.048
next1_cantidad_letras       0.046
es_funcional                0.044
prev2_ca

In [29]:
# Párrafos sin puntuación: solo vemos la predicción
parrafos = [
    "cuándo salimos para la uba ya se hace tarde por dónde andarán",
    "señor presidente quiero agradecer a la comisión por su trabajo creo que es un informe muy completo",
    "qué hora es no sé pero ya deberíamos irnos",
]
for parrafo in parrafos:
    print(f"{parrafo}\n  → {etiquetar_parrafo(modelos_rf, predecir_pipeline_clasico, parrafo)}\n")

cuándo salimos para la uba ya se hace tarde por dónde andarán
  → ¿Cuándo salimos para la uba, Ya se hace tarde ¿Por dónde andarán?

señor presidente quiero agradecer a la comisión por su trabajo creo que es un informe muy completo
  → Señor Presidente, Quiero agradecer a la Comisión por su trabajo. Creo que es un informe. muy completo.

qué hora es no sé pero ya deberíamos irnos
  → ¿Qué hora, Es no sé, pero ya deberíamos irnos?



### Comparación con párrafos reales

In [30]:
# Párrafo de ejemplo con su puntuación real: lo normalizamos, lo predecimos y comparamos
parrafo_ejemplo = (
    "Señor Presidente, quiero agradecer a la Comisión por su trabajo. "
    "Creo que es un informe muy completo, pero ¿qué va a pasar con los fondos de la UE? "
    "Espero una respuesta clara."
)
comparar_parrafo(modelos_rf, predecir_pipeline_clasico, parrafo_ejemplo)

Real:     Señor Presidente, quiero agradecer a la Comisión por su trabajo. Creo que es un informe muy completo, pero ¿qué va a pasar con los fondos de la UE? Espero una respuesta clara.
Predicho: Señor Presidente, Quiero agradecer a la Comisión por su trabajo. Creo que es un Informe. MUY completo, ¿pero ¿qué va a pasar con los fondos de la UE. espero. una respuesta, clara.


,palabra,punt_final (real),punt_final (pred.),capitalizacion (real),capitalizacion (pred.),punt_inicial (real),punt_inicial (pred.),errores
0,señor,,,1,1,,,
1,presidente,",",",",1,1,,,
2,quiero,,,0,1,,,capitalizacion
3,agradecer,,,0,0,,,
4,a,,,0,0,,,
5,la,,,0,0,,,
6,comisión,,,1,1,,,
7,por,,,0,0,,,
8,su,,,0,0,,,
9,trabajo,.,.,0,0,,,


# Consulta
- Sobre añadir datos sinteticos para tener más preguntas ('¿' '?').
- Sobre concatenar oraciones. (Para que el modelo tenga que separarlas) (En parte, hay que dejar oraciones únicas).
  
- Sobre parseo de oraciones y balanceo de clases
- sobre multiclases, que nos conviene?

# TODO
- chequear balanceo de clases
- realizar dt y svm o knn
- falta atributo de signo de pregunta en el texto